In [8]:
import warnings
warnings.filterwarnings("ignore")

# Data manipulation and analysis
import numpy as np
import pandas as pd

# Multi-dimensional arrays and datasets (e.g., NetCDF, Zarr)
import xarray as xr

from scipy.spatial import cKDTree

# Planetary Computer tools for STAC API access and authentication
import pystac_client
import planetary_computer as pc

from datetime import date
from tqdm import tqdm
import os

# Resolve project root when running from Our Notebooks/
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

In [ ]:
!pip install numpy pandas xarray scipy tqdm pystac-client planetary-computer zarr fsspec adlfs

In [9]:
def load_terraclimate_dataset():
    catalog = pystac_client.Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=pc.sign_inplace,
    )
    collection = catalog.get_collection("terraclimate")
    asset = collection.assets["zarr-abfs"]

    if "xarray:storage_options" in asset.extra_fields:
        ds = xr.open_zarr(
            asset.href,
            storage_options=asset.extra_fields["xarray:storage_options"],
            consolidated=True,
        )
    else:
        ds = xr.open_dataset(
            asset.href,
            **asset.extra_fields["xarray:open_kwargs"],
        )

    return ds

In [10]:
# --- Filtering function (kept identical) ---
def filterg(ds, var):
    ds_2011_2015 = ds[var].sel(time=slice("2011-01-01", "2015-12-31"))

    df_var_append = []
    for i in tqdm(range(len(ds_2011_2015.time))):
        df_var = ds_2011_2015.isel(time=i).to_dataframe().reset_index()
        df_var_filter = df_var[
            (df_var['lat'] > -35.18) & (df_var['lat'] < -21.72) &
            (df_var['lon'] > 14.97) & (df_var['lon'] < 32.79)
        ]
        df_var_append.append(df_var_filter)

    df_var_final = pd.concat(df_var_append, ignore_index=True)
    print(f"Filtering for {var} completed")

    df_var_final['time'] = df_var_final['time'].astype(str)

    # Column mapping
    col_mapping = {"lat": "Latitude", "lon": "Longitude", "time": "Sample Date"}
    df_var_final = df_var_final.rename(columns=col_mapping)

    return df_var_final


In [11]:
# --- Climate variable assignment function (unchanged logic) ---
def assign_nearest_climate(sa_df, climate_df, var_name):
    """
    Map nearest climate variable values to a new DataFrame 
    containing only the specified variable column.
    """
    sa_coords = np.radians(sa_df[['Latitude', 'Longitude']].values)
    climate_coords = np.radians(climate_df[['Latitude', 'Longitude']].values)

    tree = cKDTree(climate_coords)
    dist, idx = tree.query(sa_coords, k=1)

    nearest_points = climate_df.iloc[idx].reset_index(drop=True)

    sa_df = sa_df.reset_index(drop=True)
    sa_df[['nearest_lat', 'nearest_lon']] = nearest_points[['Latitude', 'Longitude']]

    sa_df['Sample Date'] = pd.to_datetime(sa_df['Sample Date'], dayfirst=True, errors='coerce')
    climate_df['Sample Date'] = pd.to_datetime(climate_df['Sample Date'], dayfirst=True, errors='coerce')

    climate_values = []

    for i in tqdm(range(len(sa_df)), desc=f"Mapping {var_name.upper()} values"):
        sample_date = sa_df.loc[i, 'Sample Date']
        nearest_lat = sa_df.loc[i, 'nearest_lat']
        nearest_lon = sa_df.loc[i, 'nearest_lon']

        subset = climate_df[
            (climate_df['Latitude'] == nearest_lat) &
            (climate_df['Longitude'] == nearest_lon)
        ]

        if subset.empty:
            climate_values.append(np.nan)
            continue

        nearest_idx = (subset['Sample Date'] - sample_date).abs().idxmin()
        climate_values.append(subset.loc[nearest_idx, var_name])

    output_df = pd.DataFrame({var_name: climate_values})

    
    return output_df

In [12]:
Water_Quality_df = pd.read_csv(os.path.join(PROJECT_ROOT, 'water_quality_training_dataset.csv'))
Water_Quality_df.shape

(9319, 6)

In [ ]:
# Load TerraClimate dataset and extract all core variables
# (from the dataset overview: NOT feature engineering)
tc_vars = [
    'pet', 'aet', 'def', 'q', 'ppt', 'soil', 'swe',
    'srad', 'tmax', 'tmin', 'vap', 'vpd', 'ws', 'pdsi'
]

ds = load_terraclimate_dataset()

Validation_df = pd.read_csv(os.path.join(PROJECT_ROOT, 'submission_template.csv'))

Terraclimate_training_df = Water_Quality_df[['Latitude', 'Longitude', 'Sample Date']].copy()
Terraclimate_validation_df = Validation_df[['Latitude', 'Longitude', 'Sample Date']].copy()

for var in tc_vars:
    tc_parameter = filterg(ds, var)
    Terraclimate_training_df[var] = assign_nearest_climate(Water_Quality_df, tc_parameter, var)[var]
    Terraclimate_validation_df[var] = assign_nearest_climate(Validation_df, tc_parameter, var)[var]

In [ ]:
train_path = os.path.join(PROJECT_ROOT, 'New Datasets', 'terraclimate_features_training_allbands.csv')
val_path = os.path.join(PROJECT_ROOT, 'New Datasets', 'terraclimate_features_validation_allbands.csv')

Terraclimate_training_df.to_csv(train_path, index=False)
Terraclimate_validation_df.to_csv(val_path, index=False)

In [ ]:
# Preview File
Terraclimate_training_df.head()

In [ ]:
Validation_df.head()

In [ ]:
Validation_df.shape

In [ ]:
Terraclimate_validation_df.shape

In [13]:
Terraclimate_validation_df.head()